# 심화 미션: 전기차 배터리 셀 최종검사
- 상황: 놓친 불합격 하나가 리콜로 이어지는 현장이다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 배터리 검사에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 개방 전압 | 아무것도 연결하지 않은 상태에서 잰 전압 (V) |
| 용량 | 이 셀이 담을 수 있는 전기의 양 (mAh). 클수록 오래 간다 |
| 내부 저항 | 전기가 흐를 때 셀 안에서 생기는 저항 (mΩ). 높을수록 열이 나고 성능이 떨어진다 |
| 두께 부풀음 | 셀이 규격보다 두꺼워지는 것. 안에서 가스가 생겼다는 신호일 수 있다 |
| 완충 시간 | 다 채우는 데 걸린 시간 (분). 오래 걸릴수록 어딘가 문제가 있을 수 있다 |
| 합격 / 불합격 | 검사실에서 붙이는 최종 판정 |

## Q1. 파일 열고 크기 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day04_battery.csv")

In [2]:
print(df.shape)

(2847, 14)


In [3]:
불합격수 = (df["result"] == "불합격").sum()

print(f"불합격 {불합격수}건 ({round(불합격수 / len(df) * 100, 1)}%)")

불합격 133건 (4.7%)


## Q2. 빈칸 찾아 채우기

In [4]:
# 빈칸(결측치)이 있는 열과 그 개수를 확인한다
빈칸개수 = df.isnull().sum()
빈칸있는열 = 빈칸개수[빈칸개수 > 0]

print("빈칸이 있는 열:")
print(빈칸있는열)

# 빈칸이 있는 열만 그 열의 중앙값으로 채운다
for c in 빈칸있는열.index:
    df[c] = df[c].fillna(df[c].median())

# 채운 뒤 다시 확인 - 전부 0이어야 한다
print("\n채운 뒤 빈칸 개수:")
print(df[빈칸있는열.index].isnull().sum())

빈칸이 있는 열:
charge_time_min          38
chamber_temp_C           57
chamber_humidity_pct    113
dtype: int64

채운 뒤 빈칸 개수:
charge_time_min         0
chamber_temp_C          0
chamber_humidity_pct    0
dtype: int64


## Q3.입력과 정답 가르기

In [5]:
# result가 "불합격"이면 1, "합격"이면 0인 정답 열을 만든다
df["불합격여부"] = (df["result"] == "불합격").astype(int)

# 셀 번호·검사 시각·라인·조·검사원·판정 열은 입력에서 뺀다. 나머지 숫자 검사 항목만 입력으로 쓴다
제외열 = ["cell_id", "inspected_at", "line", "shift", "inspector", "result", "불합격여부"]
검사항목열 = [c for c in df.columns if c not in 제외열]

X = df[검사항목열]
y = df["불합격여부"]

print("입력 열:", 검사항목열)
print("입력:", X.shape)
print(y.value_counts())

입력 열: ['open_voltage_V', 'capacity_mAh', 'internal_resistance_mOhm', 'thickness_mm', 'weight_g', 'charge_time_min', 'chamber_temp_C', 'chamber_humidity_pct']
입력: (2847, 8)
불합격여부
0    2714
1     133
Name: count, dtype: int64


## Q4. 학습용과 시험용으로 나누기

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 시험용으로 20% 떼어둠
    random_state=42,    # 고정 번호 - 다시 돌려도 같게 나오도록
    stratify=y           # 불합격 비율을 양쪽에 맞춰서 나눔
)

print("학습용:", X_train.shape, " 불합격:", y_train.sum(),
      f"({round(y_train.mean() * 100, 2)}%)")
print("시험용:", X_test.shape, " 불합격:", y_test.sum(),
      f"({round(y_test.mean() * 100, 2)}%)")

학습용: (2277, 8)  불합격: 106 (4.66%)
시험용: (570, 8)  불합격: 27 (4.74%)


## Q5. 기준 모델 세우기

In [7]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(합격)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")
print("기준 모델이 불합격이라 한 건수:", 기준예측.sum())

기준 모델 정확도: 95.26 %
기준 모델이 불합격이라 한 건수: 0


## Q6. 손대지 않은 모델로 한번

In [8]:
# 표준화와 로지스틱 회귀, 채점 도구를 불러온다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 열마다 단위가 다르므로, 학습용 기준으로 평균·표준편차를 구해 두 데이터 모두에 적용한다
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 아직 아무 설정도 손대지 않은 기본 상태 (class_weight 등 불균형 보정 없음)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

맞힌합격, 헛경보, 놓친불합격, 잡은불합격 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round((예측 == y_test).mean() * 100, 2), "%")
print("잡은 불합격:", 잡은불합격, "/ 놓친 불합격:", 놓친불합격, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 예측), 3),
      "정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 예측), 3))

정확도: 97.02 %
잡은 불합격: 12 / 놓친 불합격: 15 / 헛경보: 2
재현율: 0.444 정밀도: 0.857 F1: 0.585


## Q7. 드문 쪽에 무게를 주고 다시

In [9]:
# class_weight="balanced" - 드문 쪽(불합격) 한 건을 더 무겁게 세라는 뜻. 그 외 설정은 Q6과 동일
가중치모델 = LogisticRegression(max_iter=1000, class_weight="balanced")
가중치모델.fit(X_train_scaled, y_train)

가중치예측 = 가중치모델.predict(X_test_scaled)

맞힌합격_w, 헛경보_w, 놓친불합격_w, 잡은불합격_w = confusion_matrix(y_test, 가중치예측).ravel()

print("정확도:", round((가중치예측 == y_test).mean() * 100, 2), "%")
print("잡은 불합격:", 잡은불합격_w, "/ 놓친 불합격:", 놓친불합격_w, "/ 헛경보:", 헛경보_w)
print("재현율:", round(recall_score(y_test, 가중치예측), 3),
      "정밀도:", round(precision_score(y_test, 가중치예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 가중치예측), 3))

# Q6(손 안 댐)과 Q7(가중치)을 두 줄짜리 표로 나란히 정리
Q6Q7_비교표 = pd.DataFrame({
    "처리": ["손 안 댐", "가중치"],
    "정확도(%)": [round((예측 == y_test).mean() * 100, 2),
                round((가중치예측 == y_test).mean() * 100, 2)],
    "잡은 불합격": [잡은불합격, 잡은불합격_w],
    "놓친 불합격": [놓친불합격, 놓친불합격_w],
    "헛경보": [헛경보, 헛경보_w],
    "재현율": [round(recall_score(y_test, 예측), 3),
              round(recall_score(y_test, 가중치예측), 3)],
    "정밀도": [round(precision_score(y_test, 예측, zero_division=0), 3),
              round(precision_score(y_test, 가중치예측, zero_division=0), 3)],
    "F1": [round(f1_score(y_test, 예측), 3),
           round(f1_score(y_test, 가중치예측), 3)],
})
Q6Q7_비교표

정확도: 86.32 %
잡은 불합격: 25 / 놓친 불합격: 2 / 헛경보: 76
재현율: 0.926 정밀도: 0.248 F1: 0.391


,처리,정확도(%),잡은 불합격,놓친 불합격,헛경보,재현율,정밀도,F1
0,손 안 댐,97.02,12,15,2,0.444,0.857,0.585
1,가중치,86.32,25,2,76,0.926,0.248,0.391


## Q8. 다이얼 세 번 돌리기

In [10]:
# 나무 모델을 불러온다 (표준화는 트리 모델에는 필요 없다)
from sklearn.tree import DecisionTreeClassifier

결과_깊이 = []

# 세 가지 깊이를 차례로 넣어본다. None은 제한 없이 끝까지 간다는 뜻
for 깊이 in [3, 5, None]:
    나무 = DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=깊이)
    나무.fit(X_train, y_train)
    나무예측 = 나무.predict(X_test)

    맞힌합격_t, 헛경보_t, 놓친불합격_t, 잡은불합격_t = confusion_matrix(y_test, 나무예측).ravel()

    결과_깊이.append({
        "깊이": 깊이 if 깊이 is not None else "제한 없음",
        "정확도(%)": round((나무예측 == y_test).mean() * 100, 2),
        "잡은 불합격": 잡은불합격_t,
        "헛경보": 헛경보_t,
        "재현율": round(recall_score(y_test, 나무예측), 3),
        "F1": round(f1_score(y_test, 나무예측), 3),
    })

깊이_비교표 = pd.DataFrame(결과_깊이)
깊이_비교표

,깊이,정확도(%),잡은 불합격,헛경보,재현율,F1
0,3,88.42,24,63,0.889,0.421
1,5,85.79,19,73,0.704,0.319
2,제한 없음,92.98,11,24,0.407,0.355


## Q9. 자동 탐색으로 설정하기

In [11]:
# 후보 조합을 자동으로 돌려보는 도구와 겹 나누는 도구를 불러온다
from sklearn.model_selection import GridSearchCV, StratifiedKFold

파라미터_후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
}

# 다섯 겹으로 나누되 섞어서, 겹 나누기도 고정 번호 42
겹나누기 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=파라미터_후보,
    scoring="f1",
    cv=겹나누기,
)

# 학습용만 넣는다. 시험용은 이 탐색에 전혀 쓰지 않는다
탐색.fit(X_train, y_train)

print("1등 설정값:", 탐색.best_params_)
print("탐색 중 나온 점수(교차검증 F1):", round(탐색.best_score_, 3))

# 1등 설정으로 이미 학습용 전체에 다시 학습된 모델
최적모델 = 탐색.best_estimator_
최적예측 = 최적모델.predict(X_test)

맞힌합격_g, 헛경보_g, 놓친불합격_g, 잡은불합격_g = confusion_matrix(y_test, 최적예측).ravel()

print("정확도:", round((최적예측 == y_test).mean() * 100, 2), "%")
print("잡은 불합격:", 잡은불합격_g, "/ 놓친 불합격:", 놓친불합격_g, "/ 헛경보:", 헛경보_g)
print("재현율:", round(recall_score(y_test, 최적예측), 3),
      "정밀도:", round(precision_score(y_test, 최적예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 최적예측), 3))

1등 설정값: {'max_depth': 10, 'min_samples_leaf': 1}
탐색 중 나온 점수(교차검증 F1): 0.38
정확도: 91.4 %
잡은 불합격: 15 / 놓친 불합격: 12 / 헛경보: 37
재현율: 0.556 정밀도: 0.288 F1: 0.38


## Q10. 교차검증으로 마무리

In [12]:
# 한 번에 여러 지표를 재는 도구와 파이프라인 도구를 불러온다
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

# 세 모델. 겹나누기(Q9에서 만든 다섯 겹, 고정 번호 42)를 그대로 재사용한다
모델_목록 = {
    "로지스틱 (기본)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "로지스틱 (가중치)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "나무 (Q9 최적)": DecisionTreeClassifier(random_state=42, class_weight="balanced", **탐색.best_params_),
}

결과_교차검증 = []

for 이름, 모델 in 모델_목록.items():
    # 학습용 안에서만 다섯 번 재고, 재현율과 F1 두 가지를 한 번에 받는다
    점수모음 = cross_validate(모델, X_train, y_train, cv=겹나누기, scoring=["recall", "f1"])

    결과_교차검증.append({
        "모델": 이름,
        "재현율 평균": round(점수모음["test_recall"].mean(), 3),
        "재현율 흔들림": round(점수모음["test_recall"].std(), 3),
        "F1 평균": round(점수모음["test_f1"].mean(), 3),
        "F1 흔들림": round(점수모음["test_f1"].std(), 3),
    })

세모델_비교표 = pd.DataFrame(결과_교차검증)
세모델_비교표

,모델,재현율 평균,재현율 흔들림,F1 평균,F1 흔들림
0,로지스틱 (기본),0.406,0.064,0.517,0.085
1,로지스틱 (가중치),0.859,0.064,0.385,0.029
2,나무 (Q9 최적),0.500,0.067,0.380,0.050


### 과장님께 드리는 권고

1. 놓친 불합격 하나가 리콜로 이어지는 현장이니 **가중치를 준 로지스틱 회귀**를 권합니다 — 교차검증 재현율 평균이 0.859로 세 모델 중 가장 높고, 흔들림(0.064)도 나무 모델과 비슷한 수준으로 안정적입니다.
2. 손대지 않은 로지스틱 회귀(재현율 0.406)와 Q9 나무(재현율 0.500)는 F1은 더 높지만 정작 불합격을 놓치는 비율이 훨씬 커서, 리콜 리스크가 큰 이 공정에는 맞지 않습니다.
3. 다만 가중치 모델은 정밀도가 낮아 헛경보가 늘어나므로, 곧바로 자동 폐기하기보다는 이 모델이 불합격으로 지목한 건만 사람이 한 번 더 재검수하는 단계를 함께 두는 것을 권합니다.